# 다봐요 (Dabwayo) — GPU Generation Server (Google Colab, free T4)

Runs an **open-weight text→video / image→video model** on Colab's free GPU and
exposes it over a public URL. Dabwayo's `remote` provider calls it, so
`generate_video(..., provider='remote')` produces real footage that composites
into your timeline.

### Free-T4 memory notes (important)
The free tier is ~**16 GB VRAM** + ~**12.7 GB system RAM**, and the T4 GPU has
**no bfloat16** — so this notebook uses **fp16**, an **8-bit text encoder**,
**sequential CPU offload** and **VAE tiling**. If the big model still won't
load, set `MODEL = 'light'` in cell 3 (or it falls back automatically).

**Steps:** Runtime → Change runtime type → **GPU (T4)**, then Run all. Copy the
printed `DABWAYO_VIDEOGEN_URL` and set it where Dabwayo runs:
```bash
export DABWAYO_VIDEOGEN_URL='https://xxxx.trycloudflare.com'
export DABWAYO_VIDEOGEN_PROVIDER=remote
```
Keep this tab running — closing it stops the server.

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU! Runtime -> Change runtime type -> GPU'

## 2. Install dependencies

In [ ]:
%pip -q install "diffusers>=0.32" "transformers>=4.44" accelerate safetensors \
    bitsandbytes imageio imageio-ffmpeg sentencepiece fastapi "uvicorn[standard]" \
    nest-asyncio pillow
print('deps installed')

## 3. Load the model (low-memory, fp16)
`MODEL='ltx'` = LTX-Video (photoreal, t2v **and** i2v) with 8-bit text encoder
+ sequential offload + VAE tiling. `MODEL='light'` = Zeroscope (t2v only, very
light). On any load error the notebook **auto-falls back to 'light'**.

In [ ]:
import torch, gc
MODEL = 'ltx'   # 'ltx' (t2v+i2v, photoreal) or 'light' (zeroscope, t2v only)
DTYPE = torch.float16   # T4 has NO bfloat16
BACKEND = None
t2v = i2v = None

def _free():
    gc.collect();  torch.cuda.empty_cache()

def load_ltx():
    global t2v, i2v, BACKEND
    from diffusers import LTXPipeline, LTXImageToVideoPipeline
    from transformers import T5EncoderModel, BitsAndBytesConfig
    mid = 'Lightricks/LTX-Video'
    # 8-bit T5-XXL encoder: ~9GB fp16 -> ~2.5GB, the usual free-Colab OOM fix
    te = T5EncoderModel.from_pretrained(
        mid, subfolder='text_encoder',
        quantization_config=BitsAndBytesConfig(load_in_8bit=True), torch_dtype=DTYPE)
    t2v = LTXPipeline.from_pretrained(mid, text_encoder=te, torch_dtype=DTYPE)
    t2v.enable_sequential_cpu_offload()
    try: t2v.vae.enable_tiling(); t2v.vae.enable_slicing()
    except Exception: pass
    i2v = LTXImageToVideoPipeline(**t2v.components)   # shares modules, no extra RAM
    BACKEND = 'ltx'

def load_light():
    global t2v, i2v, BACKEND
    from diffusers import DiffusionPipeline
    t2v = DiffusionPipeline.from_pretrained('cerspense/zeroscope_v2_576w', torch_dtype=DTYPE)
    t2v.enable_model_cpu_offload()
    try: t2v.enable_vae_slicing()
    except Exception: pass
    i2v = None   # zeroscope is text-to-video only
    BACKEND = 'light'

try:
    (load_ltx if MODEL == 'ltx' else load_light)()
except Exception as e:
    print('Primary model failed -> falling back to light:', repr(e)[:200])
    t2v = i2v = None; _free()
    load_light()
print('READY backend =', BACKEND, '| modes =', ['t2v'] + (['i2v'] if i2v else []))

## 4. Generation API
Implements the contract Dabwayo's `RemoteHTTPProvider` speaks:
`GET /health` and `POST /generate` → raw `video/mp4`.

In [ ]:
import base64, io, tempfile
from PIL import Image
from fastapi import FastAPI, Request, Response, HTTPException
from diffusers.utils import export_to_video

def _snap(v, m, lo): return max(lo, int(round(v / m)) * m)

def run_generation(body: dict) -> bytes:
    prompt = body.get('prompt', '')
    mode = body.get('mode', 't2v')
    fps = float(body.get('fps', 24))
    steps = int(body.get('steps', 30))
    seed = body.get('seed')
    gen = torch.Generator(device='cuda').manual_seed(int(seed)) if seed is not None else None
    # keep defaults modest so inference fits a free T4
    if BACKEND == 'ltx':
        w = _snap(int(body.get('width', 704)), 32, 32)
        h = _snap(int(body.get('height', 480)), 32, 32)
        nf = max(9, int(round((int(body.get('num_frames', 73)) - 1) / 8)) * 8 + 1)
        kw = dict(prompt=prompt, width=w, height=h, num_frames=nf,
                  num_inference_steps=steps,
                  guidance_scale=float(body.get('guidance', 3.0)), generator=gen)
        if mode == 'i2v':
            if i2v is None: raise HTTPException(400, 'i2v not supported by this backend')
            if not body.get('image_b64'): raise HTTPException(400, 'i2v needs image_b64')
            img = Image.open(io.BytesIO(base64.b64decode(body['image_b64']))).convert('RGB')
            frames = i2v(image=img.resize((w, h)), **kw).frames[0]
        else:
            frames = t2v(**kw).frames[0]
    else:  # zeroscope (light)
        if mode == 'i2v': raise HTTPException(400, "i2v needs MODEL='ltx'")
        w = _snap(int(body.get('width', 576)), 8, 256)
        h = _snap(int(body.get('height', 320)), 8, 256)
        nf = int(body.get('num_frames', 24))
        frames = t2v(prompt, num_frames=nf, height=h, width=w,
                     num_inference_steps=steps, generator=gen).frames[0]
    path = tempfile.mktemp(suffix='.mp4')
    export_to_video(frames, path, fps=fps)
    _free()
    return open(path, 'rb').read()

app = FastAPI()

@app.get('/health')
def health():
    return {'ok': True, 'backend': BACKEND,
            'modes': ['t2v'] + (['i2v'] if i2v else []),
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}

@app.post('/generate')
async def generate(request: Request):
    body = await request.json()
    try:
        return Response(content=run_generation(body), media_type='video/mp4')
    except torch.cuda.OutOfMemoryError:
        _free(); raise HTTPException(507, 'GPU OOM — lower num_frames/width/height')

print('API defined')

## 5. Public tunnel + launch
Starts the server and a free Cloudflare quick-tunnel (no account needed).

In [ ]:
import nest_asyncio, threading, uvicorn, subprocess, re, time, os, urllib.request, stat
nest_asyncio.apply()

def _serve(): uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')
threading.Thread(target=_serve, daemon=True).start()
time.sleep(3)

BIN = '/usr/local/bin/cloudflared'
if not os.path.exists(BIN):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', BIN)
    os.chmod(BIN, os.stat(BIN).st_mode | stat.S_IEXEC)

proc = subprocess.Popen([BIN, 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    m = re.search(r'https://[\w.-]+\.trycloudflare\.com', line)
    if m: url = m.group(0); break

print('\n' + '=' * 60)
print('DABWAYO_VIDEOGEN_URL =', url)
print('=' * 60)
print('On the machine running Dabwayo:')
print(f"  export DABWAYO_VIDEOGEN_URL='{url}'")
print('  export DABWAYO_VIDEOGEN_PROVIDER=remote')
print('Keep this tab open to keep the server alive.')

## 6. (Optional) Smoke test

In [ ]:
import requests
print('health:', requests.get(url + '/health', timeout=30).json())
r = requests.post(url + '/generate', json={'prompt': 'a golden retriever running on a beach, slow motion',
    'mode': 't2v', 'num_frames': 49, 'width': 640, 'height': 384, 'fps': 24, 'steps': 24}, timeout=1200)
open('test.mp4', 'wb').write(r.content); print('wrote test.mp4', len(r.content), 'bytes')
from IPython.display import Video; Video('test.mp4', embed=True)